# cachetools experiment

Python utilities for in-memory TTL and LRU caches. This notebook demonstrates one consistent sample-document RAG and agent-memory workflow.

In [ ]:
# Install: pip install cachetools
# Setup: No cache service is required; run `pip install cachetools`. This notebook configures a one-hour process-local TTL.
print('Install the package and start the required service, if any.')

## Using cachetools for RAG and agentic AI

**Why:** It is lightweight for caching retrieval results inside one Python process.

**When:** Choose cachetools for local prototypes or per-worker caches where losing entries on restart is acceptable.

**Limit:** Avoid it when multiple workers must share entries or when cache data must persist across restarts.

In [ ]:
from cachetools import TTLCache
# Process-local; not shared across workers.
cache = TTLCache(maxsize=1000, ttl=3600)
def cache_get(key):
    return cache.get(key)
def cache_set(key, value):
    cache[key] = value
def cache_delete(key):
    return cache.pop(key, None) is not None

## 1. RAG workflow

Search the same small document in every cache-tool experiment. On a miss, the selected cache tool stores the retrieved passage; rerunning the cell should read it from that tool.

In [ ]:
import hashlib
import re

sample_document = 'RAG retrieves relevant passages from documents before answering a question.\nA cache stores frequently requested RAG retrieval results so repeated questions return faster.\nAgent memory stores task progress and tool results so later steps can continue the task.\nRedis stores values by key and can expire them automatically using a time-to-live.'
document_version = "sample-doc-v1"
question = "How can a cache speed up repeated RAG retrieval?"

def retrieve_relevant_passage(question, document):
    stop_words = {"how", "can", "a", "the", "up", "to", "and", "for", "does"}
    query_terms = set(re.findall(r"[a-z0-9]+", question.lower())) - stop_words
    passages = [line.strip() for line in document.splitlines() if line.strip()]
    ranked = sorted(
        passages,
        key=lambda passage: len(
            query_terms & set(re.findall(r"[a-z0-9]+", passage.lower()))
        ),
        reverse=True,
    )
    if not ranked or not (
        query_terms & set(re.findall(r"[a-z0-9]+", ranked[0].lower()))
    ):
        raise ValueError("No matching passage was found in the sample document.")
    return {"text": ranked[0], "source": document_version}

query_hash = hashlib.sha256(
    f"{document_version}:{question.lower().strip()}".encode()
).hexdigest()
rag_cache_key = f"rag:context:{query_hash}"

cached_context = cache_get(rag_cache_key)
if cached_context is None:
    print("CACHE MISS: retrieving from the sample document")
    retrieved = retrieve_relevant_passage(question, sample_document)
    cache_set(rag_cache_key, retrieved)
else:
    print("CACHE HIT: served by cachetools")
    retrieved = cached_context

print("Question:", question)
print("Retrieved context:", retrieved)

# Run this cell again to read the same context from the selected cache tool.

## 2. Agent session workflow

Store the retrieved passage and current task step under an agent-session key using this cache tool.

In [ ]:
session_id = 'demo-session-001'
agent_state_key = f'agent:session:{session_id}:state'
agent_state = {'step': 'retrieval_complete', 'context': retrieved}
cache_set(agent_state_key, agent_state)
print('Agent state from cache:', cache_get(agent_state_key))

## 3. Prompt and response workflow

Build a prompt from the retrieved passage, then store/read that prompt payload with the same cache tool. Call your model with this prompt in your application.

In [ ]:
prompt_key = f'agent:prompt:{document_version}:{query_hash}'
prompt_payload = {'question': question, 'context': retrieved['text']}
cache_set(prompt_key, prompt_payload)
print('Prompt payload from cache:', cache_get(prompt_key))

## 4. TTL and freshness

This backend/API does not support a per-entry TTL in this example; use the stated versioning or backend policy.

In [ ]:
freshness_key = f'{rag_cache_key}:sample-doc-v2'
cache_set(freshness_key, retrieved)
print('Versioned value:', cache_get(freshness_key))

## 5. Agent tool result workflow

The retrieved passage is the real output of the notebook's retriever. Store it as an agent tool result for reuse.

In [ ]:
tool_result_key = f'agent:tool-result:{session_id}:retriever'
cache_set(tool_result_key, retrieved)
print('Tool result from cache:', cache_get(tool_result_key))

## 6. Read or reuse the result

Read the cached RAG context again with the same key and verify the backend returns the stored passage.

In [ ]:
cached_again = cache_get(rag_cache_key)
print('Reused cached passage:', cached_again)
assert cached_again == retrieved

## 7. Invalidate stale data

Delete the RAG key after its source document changes or the cached result is no longer valid.

In [ ]:
deleted = cache_delete(rag_cache_key)
print('RAG cache entry deleted:', bool(deleted))
print('Read after deletion:', cache_get(rag_cache_key))

## 8. Application workflow

Use this same flow in your application: retrieve from the current document/index version, build a versioned key, check this tool, and on a miss store the real retrieved context. Pass that context to your LLM for generation. Agent state uses a separate session namespace.

In [ ]:
print('cachetools: sample-document RAG and agent cache example completed. Use README.md for when and why to choose this tool.')